# Structural changes — interactive visualization

This notebook provides an interactive exploratory data analysis (EDA) for the two-dimensional piecewise AR(1) stream generator in `src/simulation/structural_changes.py`. Use the widgets to change the scenario and parameters, inspect the time series, 2D joint distribution, and simple rolling diagnostics.

Recommended workflow:
- Use this notebook for interactive experimentation and to select representative seeds/parameters.
- Re-run the same parameters in `src/visualization/plot_structural_changes.py` (not included here) to produce reproducible figures saved to `results/`.


In [ ]:
# Setup: imports and path adjustments
import os, sys
# Ensure project root is on sys.path so `src` imports work from the notebook
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ipywidgets import interactive, IntSlider, FloatSlider, Dropdown, HBox, VBox, Layout

# Import the generator from the repository code
from src.simulation.structural_changes import generate_structural_stream

sns.set(style="whitegrid")
plt.rcParams.update({
    'figure.max_open_warning': 0,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
})


In [ ]:
# Helper: draw covariance ellipse for a bivariate cloud
import matplotlib.transforms as transforms
from matplotlib.patches import Ellipse

def covariance_ellipse(ax, mean, cov, n_std=1.0, facecolor='none', edgecolor='C2', alpha=0.8, **kwargs):
    # cov: 2x2 covariance matrix; mean: length-2 array
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    # width, height = 2*sqrt(eigenvalues) scaled by n_std
    width, height = 2 * n_std * np.sqrt(vals)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    ell = Ellipse(xy=mean, width=width, height=height, angle=angle, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, **kwargs)
    ax.add_patch(ell)
    return ell

def rolling_stats(stream, window=50):
    # stream: (T,2) array; return rolling mean, var, corr arrays (aligned to right)
    df = pd.DataFrame(stream, columns=['x1','x2'])
    roll_mean = df.rolling(window, min_periods=1).mean()
    roll_var = df.rolling(window, min_periods=1).var()
    roll_corr = df['x1'].rolling(window, min_periods=1).corr(df['x2'])
    return roll_mean.values, roll_var.values, roll_corr.values

def plot_stream(stream, metadata, ax=None, show=True):
    T = stream.shape[0]
    change_at = metadata.get('change_at', 0)

    fig = plt.figure(constrained_layout=False, figsize=(12, 9))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.8], width_ratios=[1, 1], hspace=0.4, wspace=0.3)

    # Time series (two stacked)
    ax_ts1 = fig.add_subplot(gs[0, :])
    ax_ts1.plot(range(T), stream[:, 0], label='feature 1')
    ax_ts1.plot(range(T), stream[:, 1], label='feature 2', alpha=0.8)
    ax_ts1.axvline(change_at, color='k', linestyle='--', label='change_at')
    ax_ts1.set_title('Time series (both features)')
    ax_ts1.legend(loc='upper right')

    # 2D scatter colored by time/regime
    ax_scatter = fig.add_subplot(gs[1, 0])
    cmap = plt.get_cmap('viridis')
    colors = np.linspace(0, 1, T)
    ax_scatter.scatter(stream[:, 0], stream[:, 1], c=colors, s=14, cmap=cmap)
    ax_scatter.axvline(0) if False else None
    ax_scatter.set_xlabel('feature 1')
    ax_scatter.set_ylabel('feature 2')
    ax_scatter.set_title('2D trajectory (coloured by time)')

    # Covariance ellipses: pre vs post (using sample cov of halves)
    ax_cov = fig.add_subplot(gs[1, 1])
    if change_at > 0 and change_at < T:
        pre = stream[:change_at]
        post = stream[change_at:]
    else:
        pre = stream[:max(1, T//2)]
        post = stream[max(1, T//2):]

    ax_cov.scatter(pre[:,0], pre[:,1], alpha=0.5, label='pre', s=10)    ax_cov.scatter(post[:,0], post[:,1], alpha=0.5, label='post', s=10)    pre_mean = pre.mean(axis=0)
    post_mean = post.mean(axis=0)
    pre_cov = np.cov(pre, rowvar=False)
    post_cov = np.cov(post, rowvar=False)
    covariance_ellipse(ax_cov, pre_mean, pre_cov, n_std=1.0, edgecolor='C0')
    covariance_ellipse(ax_cov, post_mean, post_cov, n_std=1.0, edgecolor='C1')
    ax_cov.set_title('Pre / Post covariance (1 std ellipse)')
    ax_cov.legend()

    # Rolling diagnostics: mean, var, corr
    ax_rm = fig.add_subplot(gs[2, :])
    roll_mean, roll_var, roll_corr = rolling_stats(stream, window=max(3, int(0.05 * T)))
    ax_rm.plot(roll_mean[:,0], label='rolling mean x1')
    ax_rm.plot(roll_mean[:,1], label='rolling mean x2')
    ax_rm.plot(roll_corr, label='rolling corr (x1,x2)')
    ax_rm.axvline(change_at, color='k', linestyle='--')
    ax_rm.set_title('Rolling diagnostics (mean, corr)')
    ax_rm.legend(loc='upper right')

    if show:
        plt.show()
    return fig


In [ ]:
# Interactive controls and wiring
def _make_plot(seed=1, scenario='combined', n_observations=1000, change_at=500, variance_ratio=4.0, correlation_before=0.0, correlation_after=0.7, ar_before=0.0, ar_after=0.7):
    stream, metadata = generate_structural_stream(
        seed=seed,
        scenario=scenario,
        n_observations=n_observations,
        change_at=change_at,
        variance_ratio=variance_ratio,
        correlation_before=correlation_before,
        correlation_after=correlation_after,
        ar_before=ar_before,
        ar_after=ar_after,
    )
    plot_stream(stream, metadata)

scenario_options = ['mean_shift', 'variance_shift', 'correlation_shift', 'covariance_shift', 'ar_shift', 'autocorrelation_shift', 'combined', 'default']
w = interactive(
    _make_plot,
    seed=IntSlider(min=0, max=9999, step=1, value=1, description='seed'),
    scenario=Dropdown(options=scenario_options, value='combined', description='scenario'),
    n_observations=IntSlider(min=100, max=5000, step=100, value=1000, description='T'),
    change_at=IntSlider(min=0, max=1000, step=1, value=500, description='change_at'),
    variance_ratio=FloatSlider(min=0.1, max=10.0, step=0.1, value=4.0, description='var ratio'),
    correlation_before=FloatSlider(min=-0.9, max=0.9, step=0.05, value=0.0, description='rho before'),
    correlation_after=FloatSlider(min=-0.9, max=0.9, step=0.05, value=0.7, description='rho after'),
    ar_before=FloatSlider(min=-0.9, max=0.9, step=0.05, value=0.0, description='phi before'),
    ar_after=FloatSlider(min=-0.9, max=0.9, step=0.05, value=0.7, description='phi after'),
)
display(w)


Notes:
- This notebook is intended for interactive exploration and quick visual checks.
- To save reproducible figures, add a small script under `src/visualization/` that calls `generate_structural_stream` with fixed inputs and writes outputs to `results/figures/structural_changes/...`.
- If you want, I can add that reproducible script and a small example output file layout.
